# Lesson 2: File Handling & Exception Handling

**Week 3 · Data Engineering Course**

---

**What you will learn:**
- Reading and writing text files with `open()`
- Parsing CSV files with the built-in `csv` module
- Reading and writing JSON
- Handling errors with `try/except/finally`
- Navigating the file system with `pathlib`

**Run each cell with Shift+Enter. Files written in this lesson go to a temporary location — clean them up at the end.**

---

## 2.1 Reading and Writing Text Files

The `open()` function opens a file. Always use `with open(...)` — it closes the file automatically even if an error occurs.

In [ ]:
from pathlib import Path

# Create a temporary notes file
notes_path = Path('temp_notes.txt')

# Write to a file
with open(notes_path, 'w') as f:
    f.write('Line 1: Data engineering is about moving data.\n')
    f.write('Line 2: Python is the most common language for pipelines.\n')
    f.write('Line 3: pandas makes data cleaning fast.\n')

print(f'File written: {notes_path.resolve()}')

In [ ]:
# Read the whole file at once
with open(notes_path, 'r') as f:
    content = f.read()
print(content)

In [ ]:
# Read line by line (memory-efficient for large files)
with open(notes_path, 'r') as f:
    for i, line in enumerate(f, start=1):
        print(f'Line {i}: {line.rstrip()}')  # rstrip() removes the trailing newline

---

## 2.2 The csv Module

The `csv` module handles CSV files properly — it deals with quoted fields, commas inside values, and different delimiters.

In [ ]:
import csv

# Read a CSV as a list of dictionaries (one dict per row)
csv_path = Path('../data/raw/sales_q1.csv')

rows = []
with open(csv_path, 'r', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(dict(row))

print(f'Rows loaded: {len(rows)}')
print('First row:')
for key, value in rows[0].items():
    print(f'  {key}: {repr(value)}')

In [ ]:
# Find rows with issues using pure Python
problematic = []
for row in rows:
    issues = []
    if row['total'] == '':
        issues.append('missing total')
    if row['category'] != row['category'].strip():
        issues.append('whitespace in category')
    if row['customer_name'] != row['customer_name'].strip():
        issues.append('whitespace in customer_name')
    if issues:
        problematic.append((row['order_id'], issues))

print(f'Rows with issues: {len(problematic)}')
for order_id, issues in problematic:
    print(f'  Order {order_id}: {issues}')

In [ ]:
import csv

# Write rows to a new CSV
output_path = Path('temp_sample.csv')

sample_rows = [
    {'order_id': 1001, 'product': 'Laptop', 'total': 1200.00},
    {'order_id': 1002, 'product': 'Mouse', 'total': 35.00},
]

with open(output_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['order_id', 'product', 'total'])
    writer.writeheader()
    writer.writerows(sample_rows)

print(f'Wrote {len(sample_rows)} rows to {output_path}')

---

## 2.3 JSON

JSON is the standard format for API responses and configuration files. Python's `json` module converts between JSON text and Python dicts/lists.

In [ ]:
import json

# Python dict -> JSON string
configuration = {
    'database': 'de_course',
    'host': 'localhost',
    'port': 5432,
    'tables': ['customers', 'orders', 'products']
}

json_string = json.dumps(configuration, indent=2)
print(json_string)

In [ ]:
# JSON string -> Python dict
parsed = json.loads(json_string)
print(type(parsed))             # <class 'dict'>
print(parsed['database'])       # de_course
print(parsed['tables'][0])      # customers

# Write JSON to a file
config_path = Path('temp_config.json')
with open(config_path, 'w') as f:
    json.dump(configuration, f, indent=2)

# Read JSON from a file
with open(config_path, 'r') as f:
    loaded = json.load(f)
print(f'Loaded config for: {loaded["database"]}')

---

## 2.4 Exception Handling

Data engineering code constantly touches external systems: files, databases, APIs. Any of them can fail. `try/except` lets you handle failures gracefully instead of crashing.

In [ ]:
# Without exception handling — crashes on bad input
def risky_price(price_str):
    return float(price_str)

# risky_price('N/A')   # ValueError: could not convert string to float: 'N/A'

# With exception handling
def safe_price(price_str):
    try:
        cleaned = str(price_str).replace('$', '').replace(',', '').strip()
        return float(cleaned)
    except (ValueError, TypeError) as e:
        print(f'Warning: could not parse price {repr(price_str)}: {e}')
        return None

print(safe_price('$1,200.00'))  # 1200.0
print(safe_price('N/A'))        # None (with warning)
print(safe_price(''))           # None (with warning)

In [ ]:
# try / except / else / finally
def read_csv_safe(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            rows = list(reader)
    except FileNotFoundError:
        print(f'File not found: {file_path}')
        return []
    except PermissionError:
        print(f'No permission to read: {file_path}')
        return []
    else:
        print(f'Successfully read {len(rows)} rows from {file_path}')
        return rows
    finally:
        print('read_csv_safe() complete')  # always runs

_ = read_csv_safe(Path('../data/raw/sales_q1.csv'))
_ = read_csv_safe(Path('nonexistent.csv'))

---

## 2.5 pathlib — Modern File Paths

`pathlib.Path` is the modern way to work with file paths. It is cleaner than string concatenation and works on Windows, Mac, and Linux without changes.

In [ ]:
from pathlib import Path

base = Path('../data/raw')

print(base.resolve())          # absolute path
print(base.exists())           # True if it exists
print(base.is_dir())           # True if it is a directory

# List all CSV files in the folder
csv_files = sorted(base.glob('*.csv'))
for f in csv_files:
    size_kb = f.stat().st_size / 1024
    print(f'{f.name}  ({size_kb:.1f} KB)')

In [ ]:
# Building paths safely (works on Windows and Mac)
clean_dir = Path('../data/clean')
clean_dir.mkdir(parents=True, exist_ok=True)  # create if it does not exist

output_file = clean_dir / 'sales_2023.csv'    # / joins path segments
print(output_file)

# Path parts
example = Path('/Users/student/lessons/week3/data/raw/sales_q1.csv')
print(example.stem)    # sales_q1   (filename without extension)
print(example.suffix)  # .csv
print(example.parent)  # /Users/student/lessons/week3/data/raw

In [ ]:
# Clean up temporary files created in this lesson
for temp in [Path('temp_notes.txt'), Path('temp_sample.csv'), Path('temp_config.json')]:
    if temp.exists():
        temp.unlink()
        print(f'Deleted {temp}')

---

## 2.6 Key Takeaways

1. Always use `with open(...)` — it guarantees the file is closed even if an error occurs.
2. `csv.DictReader` gives you each row as a dict keyed by the column header — much more readable than a plain list.
3. `json.loads()` parses a JSON string into Python. `json.load()` reads directly from a file.
4. `try/except` prevents crashes on bad data. **Catch specific exceptions** (ValueError, FileNotFoundError) rather than bare `except:`.
5. `pathlib.Path` replaces string path manipulation. Use `/` to join path segments — it works on every OS.
6. `Path.glob('*.csv')` finds all matching files — the starting point for any multi-file pipeline.